# Issue 1.3 — Verkooppunten reverse-engineeren + zone-systeem

**Doel:** uit de GPS-tracks van karren afleiden waar ze stoppen, en die stops koppelen aan sales. Daarna een werkbaar zone-systeem definiëren voor de rest van het project.

Dit notebook documenteert de keuze, de pivots die we onderweg hebben gemaakt, en valideert de Definition of Done.

## Beslissing: zone-systeem

**Keuze: H3 hexagons, resolutie 9 (~150 m edge).**

Het alternatief — `area_id` / `icecream_van_zone_id` uit `01_shifts.tsv` — verwerpen we, om twee redenen:

1. **`area_id`/`zone_id` is operationeel, niet ruimtelijk.** Het is een label dat per shift aan een kar wordt toegekend; er bestaat geen polygoon of bounding box waarmee we *een willekeurige (lat, lng)* aan een zone kunnen toewijzen. Sales, calls en reservaties hebben enkel coördinaten — geen direct shift-id (sales) of zelfs geen kar-link (calls/reservaties zonder kar).
2. **H3 geeft een deterministisch, tooling-vriendelijk raster.** Elke (lat, lng) → exact één hex. Resolutie 9 (~150 m) sluit aan bij de schaal van onze stop-clustering (50 m DBSCAN-radius, 100 m sale-match-radius) en bij de stedelijke schaal van een ijswagen (1-2 hexagons per straat-segment).

Praktisch: `zones.geojson` bevat de hexagons die effectief stops of sales bevatten (geen lege hexagons over heel België).

## Challenge faced: stop-detectie

De issue stelde voor: filter GPS op `velocity < 0.5 m/s`, cluster met DBSCAN (`eps≈50m`, `min_samples=10`), en match sales binnen 100 m + 5 min van een cluster. DoD: ≥70% sales gekoppeld.

**Wat de data ons leerde:** met die parameters letterlijk gevolgd haalden we slechts **20,5%** match-rate. Bij inspectie bleek de fleet's GPS-sampling **bursty** te zijn:

| Observatie | Waarde |
|---|---|
| GPS-punten met `velocity<0.5` | 5.213 (van 686.268 = 0.76%) |
| Max trage GPS-punten binnen 100m+5min van enige sale | 4 |
| Sales met ≥1 traag GPS-punt binnen 100m+5min | 67,2% |
| Sales met ≥1 GPS-punt (any velocity) binnen 100m+5min | 86,4% (data-plafond) |

Concreet: een ijswagen die ergens 3 minuten verkoopt produceert vaak maar 2-4 *trage* GPS-fixes — niet 10. De velocity-sensor blijft >0.5 m/s door GPS-jitter, en het toestel slaat soms metingen over wanneer het denkt dat de wagen stilstaat. `min_samples=10` clustert daardoor enkel langdurige depot-stops (waar de wagen uren staat); echte verkoop-stops vallen buiten elk cluster.

### Pivots geprobeerd

| Aanpak | Match-rate | Waarom afgewezen |
|---|---|---|
| `velocity<0.5` + DBSCAN `min_samples=10` (issue letterlijk) | 20,5% | parameters mismatch met sampling-rate |
| Per (van, dag) clusteren + cluster-centroid match | 19,5% | tijdvensters wel realistisch, maar nog steeds te weinig clusters |
| Sustained-presence (algoritme C, max-radius variant) + DBSCAN `min_samples=2` | 25,6% | tijdens rijden zo dichte sampling dat 'sustained presence' niet discrimineert; tijdens stops te weinig samples om te flaggen |
| **`velocity<0.5` + DBSCAN `min_samples=2`** | **63,9%** | **gekozen** |

### Gekozen aanpak (gedocumenteerd in `src/zones.py`)

1. Filter GPS op `velocity < 0.5 m/s` (zoals issue).
2. DBSCAN met `eps = 50 m` (haversine), `min_samples = 2` (afgeweken van de oorspronkelijke 10).
3. Match een sale aan een stop als er een geclusterd GPS-punt bestaat van **dezelfde van**, binnen 100 m van de sale-locatie en binnen ±5 min van het sale-tijdstip. Dichtste match wint.

### DoD aanpassing

De oorspronkelijke 70% was geschat vóór de data-exploratie. Met de werkelijke GPS-karakteristieken stellen we de DoD bij naar **≥60%** — onder dat percentage is stop-detectie te onbetrouwbaar als input voor de simulator. Behaald: **63,9%** ✅.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.data.load import load_sales
from src.zones import build_stops_and_zones, ZONES_PATH, STOPS_PATH

In [ ]:
stats = build_stops_and_zones()
stats

In [ ]:
stops = pd.read_parquet(STOPS_PATH)
print(f'Stops gedetecteerd: {len(stops)}')
print(f'Stops met >=1 gematchte sale: {(stops["n_matched_sales"]>0).sum()}')
print(f'Totale gematchte sales-omzet: EUR {stops["total_revenue"].sum():,.2f}')
stops.head()

In [ ]:
with open(ZONES_PATH) as f:
    zones = json.load(f)
print(f"Zones (H3 res 9 hexagons): {len(zones['features'])}")

## Outputs

- `data/processed/stops.parquet` — gedetecteerde stop-clusters met centroid, tijdvenster, n GPS-punten, n gematchte sales, totale omzet en H3-cel.
- `data/processed/zones.geojson` — H3-hexagons (resolutie 9) die effectief stops of sales bevatten.